<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c38-header.png" alt="Nextia Learning · Search and Ranking: Find the Best Match" width="100%">

# Solution: Filters and facets

**[Filters and facets](https://learning.nextia-ai.com/courses/search/m07/filters-and-facets/)** · Search and Ranking: Find the Best Match · Module 7, lesson 2 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** count facets from a result list, write filters that agree with their counts and never return an empty page, and see how filters and ranking work together.

| | |
|---|---|
| **Time** | About 25 minutes |
| **Needs** | An internet connection for the setup cells (about 11 MB). No account, no key and no model download. |
| **You should know** | NDCG@10, from [Ranking measures](https://learning.nextia-ai.com/courses/search/m02/ranking-measures/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1 and matplotlib 3.11.2 (bm25s 0.3.16, PyStemmer 3.1.0, sentence-transformers 6.2.0, PyTorch 2.15.0, scikit-learn 1.9.1 and LightGBM 4.7.0 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/search/m07/filters-and-facets/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/M07-L02-filters-and-facets/filters-and-facets-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas']:
    print(name, version(name))

## Setup: get the Larkfield data

Run the next cell. It downloads the files that this lesson needs (about 10 MB) into a folder `larkfield-search`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-search` (in Colab: the **Files** panel on the left), then run the cell again.

Where the data comes from: Larkfield is an invented shop. Its products, queries and relevance judgments are derived from WANDS, the Wayfair ANnotation DataSet (MIT licence). The query log with its clicks, and the prices, stock and sale flags, are **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/data/dataset-card.md) says what is real and what is made.

In [ ]:
# Setup: download the files into larkfield-search/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/"
FILES = {
    "data/catalogue.csv.gz": "569c1c4c41403d59492e4cb22be61a7e634b052f67b4d32d12a9f14e77c08c52",
    "data/queries.csv": "51e0286150a6162502faf6502e90b89e4984949ddcb64f1e83f7e1d3251d8634",
    "data/judgments.csv.gz": "6a0b896c64a0e1000fe9401bfc359ae7995732af458120963408d71e50333ec6",
    "data/recorded/rerank.csv.gz": "38971117046aefd883c159ed9a7b196a0ca88f54757a8dc35df956d39222da88",
    "compare.py": "f09c610e8e9026e0c355f693cb56cd3e178e7217c0e001a744280fb82ff867d2",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-search").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-search and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-search.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, matplotlib, and the course helper as `cp` (NDCG, recall, MRR, intervals and the comparison table). It defines a few small helpers:

- `catalogue()`, `queries()` and `judgments()` read the data: one row per product, one row per query, and `{query_id: {product_id: grade}}` with grade 2 exact, 1 partial, 0 irrelevant.
- `show(ids)` prints products with their title and class.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import compare as cp
warnings.filterwarnings("ignore")

def catalogue():
    """One row per product."""
    df = pd.read_csv("data/catalogue.csv.gz", keep_default_na=False, dtype={"title_fr": str})
    for col in ("on_sale", "in_stock"):
        df[col] = df[col].astype(str).eq("True")
    return df

def queries():
    """One row per query with judgments (train, validation and test)."""
    return pd.read_csv("data/queries.csv", keep_default_na=False)

def judgments():
    """{query_id: {product_id: grade}}: 2 exact, 1 partial, 0 irrelevant."""
    j = pd.read_csv("data/judgments.csv.gz")
    out = {}
    for q, p, g in zip(j["query_id"], j["product_id"], j["grade"]):
        out.setdefault(q, {})[p] = int(g)
    return out

def show(ids, judged=None, n=10):
    """Print products: rank, ID, title, class (and the grade when judged is given)."""
    global _CAT
    if "_CAT" not in globals():
        _CAT = catalogue().set_index("product_id")
    cat = _CAT
    names = {2: "exact", 1: "partial", 0: "irrelevant"}
    for i, p in enumerate(list(ids)[:n], 1):
        g = "" if judged is None else f"  [{names.get(judged.get(p), 'not judged')}]"
        print(f"{i:2}. {p}  {cat.loc[p, 'title'][:55]:<55}  {cat.loc[p, 'product_class'][:22]}{g}")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. One query, its results

A **facet** is a property of the products, such as the colour or the price. A **facet count** says how many of the current results have each value. A **filter** keeps only the results with the value that a person chose. In this notebook, the results are the top 100 of hybrid search for one test query, recorded by the course team. Run the next cell.

In [ ]:
cat = catalogue().set_index("product_id")
q = queries().set_index("query_id")
j = judgments()
rr = pd.read_csv("data/recorded/rerank.csv.gz")      # query_id, product_id, hybrid_rank, score
QID = "W-473"
top = list(rr[rr["query_id"] == QID].sort_values("hybrid_rank")["product_id"])
print(QID, repr(q.loc[QID, "query"]), "->", len(top), "results")
show(top, j[QID], n=5)

## 2. Count the facets of the results

The counts come from the results, not from the whole catalogue. So every value shown has at least one product: a filter on it can never return an empty page.

In [ ]:
PRICE_BANDS = [(0, 50, "under 50"), (50, 100, "50 to 100"), (100, 250, "100 to 250"),
               (250, 500, "250 to 500"), (500, 1e9, "500 and more")]

def price_band(price):
    for lo, hi, name in PRICE_BANDS:
        if lo <= price < hi:
            return name

def main_colour(colour):
    """The first colour of a value such as "blue/red"."""
    return str(colour).split("/")[0].strip()

def facets(ids, top=6):
    """{facet: [(value, count)]}: only values that the results have, so no count is 0."""
    c = cat.loc[list(ids)]
    colour = c["colour"].map(main_colour)
    colour = colour[colour != ""]
    return {
        "product class": list(c["product_class"].value_counts().head(top).items()),
        "colour": list(colour.value_counts().head(top).items()),
        "price": [(b, int(n)) for b, n in c["price"].map(price_band).value_counts()
                  .reindex([band[2] for band in PRICE_BANDS]).dropna().items()],
        "availability": [("in stock", int(c["in_stock"].sum())), ("out of stock", int((~c["in_stock"]).sum()))],
    }

for name, values in facets(top).items():
    print(f"{name}: " + ", ".join(f"{v} ({n})" for v, n in values))

## 3. Your turn: a filter that agrees with its count

Complete `apply`. Then the last line compares, for each colour, the count of the facet with the number of products that the filter keeps. The two numbers must be the same. A filter that matches "blue" with `startswith` would also keep "blue and white", and the count would lie.

In [ ]:
def apply(ids, colour=None, price=None, in_stock=False):
    """Keep the products that match every filter, in the order of the results."""
    out = []
    for p in ids:
        r = cat.loc[p]
        if colour is not None and main_colour(r["colour"]) != colour:
            continue
        if price is not None and price_band(r["price"]) != price:
            continue
        if in_stock and not r["in_stock"]:
            continue
        out.append(p)
    return out

same = [(v, n, len(apply(top, colour=v))) for v, n in facets(top)["colour"]]
print(same)

In [ ]:
expect_hash('same', same, '0f0278c73e861633')

> **Check.** Each tuple is (colour, count, products kept), and the last two numbers are equal.

## 4. A filter keeps the order of the ranking

A filter removes products; it does not rank again. So the best remaining product stays first. Here, the filter "in stock" keeps 88 of the 100 results.

In [ ]:
kept = apply(top, in_stock=True)
print(len(top), "results,", len(kept), "in stock")
print("NDCG@10 before the filter:", round(cp.ndcg(top, j[QID], 10), 3))
print("NDCG@10 after the filter: ", round(cp.ndcg(kept, j[QID], 10), 3))
print("the filter keeps the order:", kept == [p for p in top if p in set(kept)])

NDCG@10 can go up or down after a filter: it measures the remaining list against all the judged products of the query.

## 5. A facet is only as good as the data

Many products have no colour and some have no class. A filter on colour hides them all, even the right ones.

In [ ]:
no_colour = (cat["colour"].map(main_colour) == "").mean()
no_class = (cat["product_class"] == "").mean()
print(f"products with no colour: {no_colour:.3f}, with no class: {no_class:.3f}")
plt.figure(figsize=(5, 2.2))
plt.barh(["no colour", "no class"], [no_colour, no_class])
plt.xlim(0, 1); plt.xlabel("share of the 40,066 products"); plt.tight_layout(); plt.show()

In [ ]:
expect('no_colour', round(no_colour, 3), 0.648)

> **Check.** 0.648 of the products have no colour, and 0.067 have no class.

In the project: `search facets "outdoor cushion"`, and the search page (`search serve`) shows the same counts.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Zero results and business rules](https://learning.nextia-ai.com/courses/search/m07/zero-results-and-business-rules/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/search/m07/filters-and-facets/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The products, queries and judgments come from WANDS (Wayfair, MIT licence). The query log, prices, stock and sale flags are synthetic: our code made them. No real customer is in this data.